ステップ1:スコープを決める

In [0]:
#分析対象期間：直近約1ヶ月分の投球データに絞る
#pybaseballのstatcast()関数でMLB Statcastデータを取得する際の期間指定に使用
start_date = '2026-03-26'
end_date = '2026-09-30'


ステップ2:Bronze層 — 生データをそのままDeltaで保存

In [0]:
#pybaseballライブラリをインストール（MLB Statcastデータの取得に使用）
%pip install pybaseball
from pybaseball import statcast

#指定期間のMLB Statcast投球データを取得（全列がpandas DataFrameとして返される）
df = statcast(start_date, end_date)

#Bronze層は生データをそのまま保存するのが目的
#pandas DataFrameには数値・文字列・欠損値が混在するため、
#一度すべてstr型に変換してからSpark DataFrameに変換（型推論の不整合を防ぐ）
sdf_bronze = spark.createDataFrame(df.astype(str))

#カタログとスキーマが存在しない場合は作成
spark.sql("CREATE CATALOG IF NOT EXISTS mlb")
spark.sql("CREATE SCHEMA IF NOT EXISTS mlb.bronze")

#Bronze層テーブルに生データを書き込み（毎回全件上書き）
(sdf_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mlb.bronze.statcast_raw"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS row_count FROM mlb.bronze.statcast_raw"))
display(spark.sql("DESCRIBE mlb.bronze.statcast_raw"))

シルバー層の作成

In [0]:
#Silver層の作成：Bronzeの生データをクレンジング・型変換して分析可能な形にする
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType

spark.sql("CREATE SCHEMA IF NOT EXISTS mlb.silver")

#Bronze層から生データを読み込み
sdf_bronze = spark.table("mlb.bronze.statcast_raw")

#Bronze層では全列str型で保存されているため、分析で使う主要カラムを本来の型に変換
#try_castを使用：変換できない不正な値はNULLとして扱い、エラーで止めない
sdf_silver = (sdf_bronze
    .withColumn("game_date", F.expr("try_cast(game_date as date)"))
    .withColumn("release_speed", F.expr("try_cast(release_speed as double)"))
    .withColumn("launch_speed", F.expr("try_cast(launch_speed as double)"))
    .withColumn("launch_angle", F.expr("try_cast(launch_angle as double)"))
    .withColumn("hit_distance_sc", F.expr("try_cast(hit_distance_sc as double)"))
    .withColumn("release_spin_rate", F.expr("try_cast(release_spin_rate as double)"))
    .withColumn("balls", F.expr("try_cast(balls as int)"))
    .withColumn("inning", F.expr("try_cast(inning as int)"))
    #球種を大別カテゴリに分類（Fastball / Breaking Ball / Offspeed / Other / Unknown）
    .withColumn("pitch_category", F.when(F.col("pitch_type").isin("FF", "FA", "SI", "FT", "FC"), "Fastball")
        .when(F.col("pitch_type").isin("SL", "ST", "CU", "KC", "SV", "SC"), "Breaking Ball")
        .when(F.col("pitch_type").isin("CH", "FS", "FO", "EP"), "Offspeed")
        .when(F.col("pitch_type").isin("KN", "GY"), "Other")
        .otherwise("Unknown"))
    #投手・打者の所属チームを推定（Top=home投手/away打者, Bot=away投手/home打者）
    .withColumn("pitcher_team", F.when(F.col("inning_topbot") == "Top", F.col("home_team")).otherwise(F.col("away_team")))
    .withColumn("batter_team", F.when(F.col("inning_topbot") == "Top", F.col("away_team")).otherwise(F.col("home_team")))
)

#重複排除：1球ごとの一意キーは game_pk（試合ID）+ at_bat_number（打席番号）+ pitch_number（投球番号）
#データ取得時に重複が発生することがあるため、同じ投球を排除
sdf_silver = sdf_silver.dropDuplicates(["game_pk","at_bat_number","pitch_number"])

#分析上ノイズになるレコードを除去
#- pitch_typeがNULL → 球種不明のデータは分析対象外
#- release_speedが30-110 mphの範囲外 → センサー異常などの外れ値を除外
sdf_silver = sdf_silver.filter(
    (F.col("pitch_type").isNotNull()) & (F.col("release_speed").between(30, 110))  #mph換算での妥当範囲
)

#Silver層テーブルにクレンジング済みデータを書き込み
#overwriteSchema=True：列構成が変わってもスキーマを上書きして書き込み可能にする
(sdf_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("mlb.silver.statcast_clean")
)

シルバー層のテーブルの中身の確認

In [0]:
#Silver層テーブルの確認：行数と球種ごとの件数
display(spark.sql("SELECT COUNT(*) AS row_count FROM mlb.silver.statcast_clean"))
display(spark.sql("""
    SELECT pitch_type, COUNT(*) AS n
    FROM mlb.silver.statcast_clean
    GROUP BY pitch_type
    ORDER BY n DESC
"""))

ゴールド層テーブルの作成

In [0]:
#Gold層の作成：Silver層のクレンジング済みデータを投手視点・打者視点で集計
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS mlb.gold")

#Silver層からクレンジング済みデータを読み込み
sdf_silver = spark.table("mlb.silver.statcast_clean")

#投手視点の集計：投手別・球種別の平均球速・回転数・投球数
#- pitch_count: 投球数（release_speedが記録された投球のみカウント）
#- avg_release_speed: 平均球速（mph）
#- avg_release_spin_rate: 平均回転数（rpm）
#- min/max_game_date: 対象期間の最古〜最新の試合日
#- team: 最新の試合日における所属チーム（max_byで取得、トレード対応）
sdf_pitcher_summary = (sdf_silver.groupBy("pitcher","pitch_type","pitch_category")
.agg(
    F.count("release_speed").alias("pitch_count"),
    F.avg("release_speed").alias("avg_release_speed"),
    F.avg("release_spin_rate").alias("avg_release_spin_rate"),
    F.min("game_date").alias("min_game_date"),
    F.max("game_date").alias("max_game_date"),
    F.max_by("pitcher_team", "game_date").alias("team")
))

#打者視点の集計：打者別の平均Exit Velocity・平均Launch Angle・打球数
#- batted_ball_count: 打球数（launch_speedが記録された打球のみ）
#- avg_launch_speed: 平均Exit Velocity（mph）
#- avg_launch_angle: 平均打球角度（度）
#- max_exit_velocity: 最大Exit Velocity（mph）
#- min/max_game_date: 対象期間の最古〜最新の試合日
#- team: 最新の試合日における所属チーム（max_byで取得、トレード対応）
sdf_batter_summary = (sdf_silver
    .filter(F.col("launch_speed").isNotNull()) #実際に打球が発生した投球のみ
    .groupBy("batter")
    .agg(
        F.count("*").alias("batted_ball_count"),
        F.avg("launch_speed").alias("avg_launch_speed"),
        F.avg("launch_angle").alias("avg_launch_angle"),
        F.max("launch_speed").alias("max_exit_velocity"),
        F.min("game_date").alias("min_game_date"),
        F.max("game_date").alias("max_game_date"),
        F.max_by("batter_team", "game_date").alias("team")
    )
)
#Gold層テーブルに集計結果を書き込み
(sdf_pitcher_summary.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("mlb.gold.pitcher_summary"))

(sdf_batter_summary.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("mlb.gold.batter_summary"))

選手名をIDで紐付け

In [0]:
#選手名ルックアップテーブルの作成：MLBAM IDから選手名と所属チームを取得
from pybaseball import playerid_reverse_lookup
from pyspark.sql import functions as F

#Goldテーブルから投手・打者のユニークなMLBAM IDを抽出
pitcher_ids = [row.pitcher for row in spark.table("mlb.gold.pitcher_summary").select("pitcher").distinct().collect()]
batter_ids = [row.batter for row in spark.table("mlb.gold.batter_summary").select("batter").distinct().collect()]

all_ids = list(set(int(x) for x in pitcher_ids + batter_ids if x is not None))

#pybaseballのplayerid_reverse_lookupでMLBAM IDから選手名を取得
lookup_df = playerid_reverse_lookup(all_ids, key_type="mlbam")
lookup_df = lookup_df[["key_mlbam","name_first","name_last"]]

sdf_lookup = spark.createDataFrame(lookup_df)
sdf_lookup = sdf_lookup.withColumn("full_name", F.concat_ws(" ", F.col("name_first"), F.col("name_last")))

#Silver層から各選手の最新チームを取得（投手・打者両方をカバー）
sdf_silver = spark.table("mlb.silver.statcast_clean")
sdf_pitcher_team = (sdf_silver
    .select(F.col("pitcher").alias("player_id"), F.col("pitcher_team").alias("team"), F.col("game_date"))
)
sdf_batter_team = (sdf_silver
    .select(F.col("batter").alias("player_id"), F.col("batter_team").alias("team"), F.col("game_date"))
)
sdf_teams = sdf_pitcher_team.unionByName(sdf_batter_team)
sdf_latest_team = (sdf_teams
    .groupBy("player_id")
    .agg(F.max_by("team", "game_date").alias("team"))
)

#lookupに最新チームをLEFT JOIN
sdf_lookup = (sdf_lookup
    .join(sdf_latest_team, sdf_lookup.key_mlbam == sdf_latest_team.player_id, "left")
    .drop("player_id")
)

(sdf_lookup.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("mlb.gold.player_lookup"))


Goldの各集計テーブルとJOIN

In [0]:
#Gold層の集計テーブルと選手名ルックアップテーブルを読み込み
sdf_pitcher_summary = spark.table("mlb.gold.pitcher_summary")
sdf_batter_summary = spark.table("mlb.gold.batter_summary")
sdf_lookup = spark.table("mlb.gold.player_lookup")

#投手集計テーブルに選手名をLEFT JOIN（pitcher = MLBAM IDで紐付け）
sdf_pitcher_name = (sdf_pitcher_summary
    .join(sdf_lookup, sdf_pitcher_summary.pitcher == sdf_lookup.key_mlbam, "left")
    .select("full_name","pitch_type","pitch_category","pitch_count","avg_release_speed","avg_release_spin_rate","min_game_date","max_game_date",sdf_pitcher_summary["team"]))

#打者集計テーブルに選手名をLEFT JOIN（batter = MLBAM IDで紐付け）
sdf_batter_summary = (sdf_batter_summary
    .join(sdf_lookup, sdf_batter_summary.batter == sdf_lookup.key_mlbam, "left")
    .select("full_name","batted_ball_count","avg_launch_speed","avg_launch_angle","max_exit_velocity","min_game_date","max_game_date",sdf_batter_summary["team"]))

#最終的な分析用テーブルとして書き込み（選手名付き）
(sdf_pitcher_name.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("mlb.gold.pitcher_summary_name"))

(sdf_batter_summary.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("mlb.gold.batter_summary_named"))
 
